# Session 1 — How cloned contracts inflate a cross-validated score

**Teaching case: rug pull detection from smart-contract source code** (about 100 minutes)

**Learning objectives for this session**
1. Explain how clones of the same contract template inflate cross-validated scores (objective 1).
3. Compare random, feature-grouped, similarity-grouped and temporal splits (objective 3).

**Data.** 2,278 verified ERC-20 contracts on Ethereum: 1,864 reported rug pulls and 414 ordinary tokens launched in the same years. Each contract is described by 29 numbers computed from its source code (Slither findings, keyword patterns, size). `groups.csv` records which contracts share or nearly share their source code.

**Ground rules.** The labels come from public reports. "Legitimate" means *not reported and still active after a year*, not "verified safe".

In [ ]:
# Setup: works locally (run from the teaching/ folder) and in Google Colab.
import os, sys, subprocess
from pathlib import Path
if 'google.colab' in sys.modules and not Path('rugpull-static-detection').exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/fazaaf/rugpull-static-detection.git'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost==3.4.1', 'shap==0.54.1', 'scikit-learn==1.6.1', 'openpyxl'], check=True)
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent, Path('rugpull-static-detection')] if (p / 'data' / 'D_single_era_matched.csv').exists())
print('repository root:', ROOT.resolve())

In [ ]:
import numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import matthews_corrcoef
from sklearn.utils.class_weight import compute_sample_weight

D = pd.read_csv(ROOT / 'data' / 'D_single_era_matched.csv'); D['address'] = D.address.str.lower()
G = pd.read_csv(ROOT / 'data' / 'groups.csv')
D = D.merge(G.drop(columns=['label', 'year', 'solc_minor']), on='address', validate='1:1')
FEAT = list(D.loc[:, 'total_vulnerabilities':'code_length'].columns)   # the 29 features
X, y = D[FEAT], D.label.values
print(len(D), 'contracts |', y.sum(), 'rug pulls |', (y == 0).sum(), 'legitimate |', len(FEAT), 'features')

def model(seed=0):
    # the settings used in the paper (no tuning)
    return xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                             eval_metric='logloss', random_state=seed, n_jobs=4)

def fit(m, Xtr, ytr):
    return m.fit(Xtr, ytr, sample_weight=compute_sample_weight('balanced', ytr))

## Part A — How much copying is there? (15 min)

`hash_L2` is a fingerprint of the source after removing comments, whitespace and literal values. Two contracts with the same fingerprint have the same code apart from names and numbers. `sim_group_0.9` joins contracts whose code is at least 90% similar (MinHash estimate of Jaccard similarity on 5-token shingles).

In [ ]:
rug = D[D.label == 1]
print('rug pulls:', len(rug), '| distinct fingerprints:', rug.hash_L2.nunique())
fam = D['sim_group_0.9'].value_counts()
big = fam.index[0]
print('largest near-identical family:', fam.iloc[0], 'contracts,', int(D.loc[D['sim_group_0.9'] == big, 'label'].sum()), 'of them rug pulls')
print('share of all rug pulls in that family:', round(D.loc[D['sim_group_0.9'] == big, 'label'].sum() / y.sum(), 3))

*The paper reports 1,190 contracts for this family because it counts code similarity only; `sim_group_0.9` also merges contracts with identical feature values, which adds a few.*

**Question A1.** If 5-fold cross-validation splits contracts at random, how often will a test contract from the large family have a near-identical twin in the training folds? What will the model then learn?

## Part B — Four ways to split the same data (40 min)

Same model, same features, same data. Only the rule that decides which contracts may be in the same fold changes.

In [ ]:
def cv_mcc(groups=None, seed=0):
    if groups is None:
        splits = StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y)
    else:
        splits = StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(X, y, groups)
    out = []
    for tr, te in splits:
        if len(set(y[te])) < 2:      # a fold without one of the classes has no MCC; skipped, as in the paper
            continue
        p = fit(model(seed), X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
        out.append(matthews_corrcoef(y[te], (p >= 0.5).astype(int)))
    return np.array(out)

res = {
    '1 random split': cv_mcc(None),
    '2 feature-space groups': cv_mcc(D.feature_cluster.values),
    '3 source similarity >= 0.9': cv_mcc(D['sim_group_0.9'].values),
}
# 4 temporal split: train on tokens launched 2020-2021, test on 2022-2023
tr, te = D.year <= 2021, D.year >= 2022
p = fit(model(0), X[tr], y[tr]).predict_proba(X[te])[:, 1]
res['4 temporal (train 2020-21, test 2022-23)'] = np.array([matthews_corrcoef(y[te], (p >= 0.5).astype(int))])
pd.DataFrame({k: dict(mean_MCC=v.mean().round(3), sd=v.std(ddof=1).round(3) if len(v) > 1 else np.nan, folds=len(v)) for k, v in res.items()}).T

**Note.** This notebook uses one repetition of 5-fold cross-validation to save time. The paper averages five repetitions, so its numbers (0.840, 0.458, 0.412) differ slightly from yours.

When one family holds most of the rug pulls, a fold can end up with no rug pulls at all; such folds are skipped, which is why fewer than five folds may be reported.

**Question B1.** Which of the four numbers would you report to someone who wants to screen *new* tokens? Why?

**Question B2.** Feature-space grouping already puts contracts with identical feature values in the same fold. Why is that not enough?

In [ ]:
# Where does the spread under similarity grouping come from?
rows = []
for k, (tr_, te_) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=0).split(X, y, D['sim_group_0.9'].values)):
    if len(set(y[te_])) < 2:
        rows.append(dict(fold=k, largest_family_in_test=bool((D['sim_group_0.9'].values[te_] == big).any()),
                         test_rug_pulls=int(y[te_].sum()), MCC=np.nan)); continue
    p_ = fit(model(0), X.iloc[tr_], y[tr_]).predict_proba(X.iloc[te_])[:, 1]
    rows.append(dict(fold=k, largest_family_in_test=bool((D['sim_group_0.9'].values[te_] == big).any()),
                     test_rug_pulls=int(y[te_].sum()), MCC=round(matthews_corrcoef(y[te_], (p_ >= 0.5).astype(int)), 3)))
pd.DataFrame(rows)

## Part C — Transfer between families (20 min)

Train on everything **except** the largest family, then test on that family. Then do the reverse: train on the family (plus the legitimate tokens) and test on the other rug pulls.

In [ ]:
in_big = D['sim_group_0.9'].values == big
leg = y == 0
# (a) without the family -> detect the family?  Legitimate tokens split in half to keep a test set for both classes.
rng = np.random.default_rng(0); half = rng.random(len(D)) < 0.5
tr_a = (~in_big & (y == 1)) | (leg & half);  te_a = in_big & (y == 1)
pa = fit(model(0), X[tr_a], y[tr_a]).predict_proba(X[te_a])[:, 1]
# (b) only the family -> detect the other rug pulls?
tr_b = (in_big & (y == 1)) | (leg & half); te_b = ~in_big & (y == 1)
pb = fit(model(0), X[tr_b], y[tr_b]).predict_proba(X[te_b])[:, 1]
print('trained without the family, recall on the family:', round((pa >= 0.5).mean(), 3))
print('trained on the family only, recall on other rug pulls:', round((pb >= 0.5).mean(), 3))

**Note.** The paper reports 0.268 and 0.022 for a similar experiment; the exact numbers depend on which legitimate tokens are used for training.

## Wrap-up (15 min) — write down
1. In one sentence: what does a high cross-validated score of this model measure?
2. One design rule you would add to any evaluation of a code-based fraud detector.